In [8]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


In [9]:
from langchain_openai import ChatOpenAI

OPENAI_MODEL = "gpt-4o-mini" 
TEMPERATURE = 0

def get_llm(model: str = OPENAI_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    return ChatOpenAI(
        api_key=OPENAI_API_KEY,
        model=model,
        temperature=temperature,
    )

llm = get_llm()

In [10]:
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate, 
    ChatPromptTemplate
)

examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    # AI가 제공한 3번째 example임
    {
        "news": "한국은행이 기준금리를 현 수준에서 동결했다. 물가 상승세는 둔화되고 있지만 가계부채 증가에 대한 우려가 여전히 크다는 판단에 따른 결정이다.",
        "keywords": "한국은행, 기준금리, 가계부채"
    },
]


In [11]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])


In [12]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)


In [13]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 뉴스 키워드 추출 전문가입니다. 핵심 키워드 3개를 추출하세요."),
    few_shot_prompt,
    ("human", "{input}")
])


In [14]:
from langchain_core.output_parsers import StrOutputParser

chain = final_prompt | llm | StrOutputParser()

In [15]:
test_news = [
    "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.",
    # AI가 제공
    "손흥민이 프리미어리그 경기에서 멀티골을 기록하며 토트넘의 3대1 승리를 이끌었다. 이번 득점으로 손흥민은 리그 통산 득점 순위를 한 단계 끌어올렸다.",
    # AI가 제공
    "기상청은 올여름 평균 기온이 관측 이래 최고치를 기록했다고 밝혔다. 전문가들은 기후변화로 인한 폭염이 앞으로 더 잦아질 것이라며 온실가스 감축 대책이 시급하다고 지적했다.",
]

In [17]:
for idx, news in enumerate(test_news, start=1):
    print(f"{idx}. {news}")
    print(chain.invoke({"input" : news}))
    print("="*30)

1. 제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.
키워드: 제미나이 2.0, 구글 AI 스튜디오, 멀티모달 입력
2. 손흥민이 프리미어리그 경기에서 멀티골을 기록하며 토트넘의 3대1 승리를 이끌었다. 이번 득점으로 손흥민은 리그 통산 득점 순위를 한 단계 끌어올렸다.
키워드: 손흥민, 프리미어리그, 멀티골
3. 기상청은 올여름 평균 기온이 관측 이래 최고치를 기록했다고 밝혔다. 전문가들은 기후변화로 인한 폭염이 앞으로 더 잦아질 것이라며 온실가스 감축 대책이 시급하다고 지적했다.
키워드: 기상청, 기후변화, 온실가스 감축
